In [ ]:
# ==============================
# 1️⃣ Setup Workspace
# ==============================

from pathlib import Path
import os

WORKSPACE = Path("./ComfyUI")

# Clone ComfyUI if not exists
if not WORKSPACE.exists():
    print("Cloning ComfyUI...")
    !git clone https://github.com/comfyanonymous/ComfyUI {WORKSPACE}
else:
    print("ComfyUI folder already exists.")

%cd {WORKSPACE}

print("Installing dependencies...")
!pip install xformers!=0.0.18 -r requirements.txt


# ==============================
# 2️⃣ Install ComfyUI-Manager
# ==============================

MANAGER_PATH = WORKSPACE / "custom_nodes" / "ComfyUI-Manager"

if not MANAGER_PATH.exists():
    print("Installing ComfyUI-Manager...")
    !git clone https://github.com/ltdrdata/ComfyUI-Manager.git {MANAGER_PATH}
else:
    print("ComfyUI-Manager already installed.")

%cd {WORKSPACE}


# ==============================
# 3️⃣ Install Pinggy (for public URL)
# ==============================

!pip install pinggy


# ==============================
# 4️⃣ Launch ComfyUI + Tunnel
# ==============================

import subprocess
import threading
import time
import socket
import pinggy

def pinggy_tunnel_thread(port):
    print("Waiting for ComfyUI server to start...")

    while True:
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        result = sock.connect_ex(('127.0.0.1', port))
        sock.close()
        if result == 0:
            break

    print("\nComfyUI loaded. Starting Pinggy tunnel...")

    try:
        tunnel = pinggy.start_tunnel(forwardto=f"localhost:{port}")
        print("\n==========================================")
        print(f"ACCESS URL: {tunnel.urls[0]}")
        print("==========================================")
        while True:
            time.sleep(60)
    except Exception as e:
        print(f"Pinggy error: {e}")


In [ ]:
!ls {WORKSPACE}/custom_nodes

In [ ]:
%cd custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
!pip install -r requirements.txt
%cd {WORKSPACE}

In [ ]:
from pathlib import Path
WORKSPACE = Path("./ComfyUI")

# Create model directories if they don't exist
(WORKSPACE / "models" / "unet").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "models" / "loras").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "models" / "clip").mkdir(parents=True, exist_ok=True)
(WORKSPACE / "models" / "vae").mkdir(parents=True, exist_ok=True)

# UNET Models
!wget -c "https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/HighNoise/Wan2.2-I2V-A14B-HighNoise-Q2_K.gguf" \
-O {WORKSPACE}/models/unet/Wan2.2-I2V-A14B-HighNoise-Q2_K.gguf

!wget -c "https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-I2V-A14B-LowNoise-Q2_K.gguf" \
-O {WORKSPACE}/models/unet/Wan2.2-I2V-A14B-LowNoise-Q2_K.gguf

# LORA Models
!wget -c "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors" \
-O {WORKSPACE}/models/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_high_noise.safetensors

!wget -c "https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors" \
-O {WORKSPACE}/models/loras/wan2.2_i2v_lightx2v_4steps_lora_v1_low_noise.safetensors

!wget -c "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/v2.0/SVI_v2_PRO_Wan2.2-I2V-A14B_HIGH_lora_rank_128_fp16.safetensors" \
-O {WORKSPACE}/models/loras/SVI_v2_PRO_Wan2.2-I2V-A14B_HIGH_lora_rank_128_fp16.safetensors

!wget -c "https://huggingface.co/Kijai/WanVideo_comfy/resolve/main/LoRAs/Stable-Video-Infinity/v2.0/SVI_v2_PRO_Wan2.2-I2V-A14B_LOW_lora_rank_128_fp16.safetensors" \
-O {WORKSPACE}/models/loras/SVI_v2_PRO_Wan2.2-I2V-A14B_LOW_lora_rank_128_fp16.safetensors

!wget -c "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors" \
-O {WORKSPACE}/models/clip/umt5_xxl_fp8_e4m3fn_scaled.safetensors

!wget -c "https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/vae/wan_2.1_vae.safetensors" \
-O /kaggle/working/ComfyUI/models/vae/wan_2.1_vae.safetensors

In [ ]:
!pkill -f main.py
# Wait a second for the port to release
import time
time.sleep(2)

tunnel_thread = threading.Thread(target=pinggy_tunnel_thread, daemon=True, args=(8188,))
tunnel_thread.start()

# Launch ComfyUI
#get_ipython().system_raw('python main.py --lowvram > comfyui.log 2>&1 &')

print("Waiting for tunnel to initialize...")
time.sleep(5) # Give the system a moment to start the log file

#get_ipython().system_raw('python main.py > comfyui.log 2>&1 &')
get_ipython().system_raw('python main.py --lowvram --fp16-vae> comfyui.log 2>&1 &')

In [ ]:
!tail -f comfyui.log